# Image Description Extraction using Mistral's Vision Models

In this notebook, we'll use the `Mistral` API to extract structured image descriptions in JSON format using the `ministral-14b-2512` model. We'll send an image URL and prompt the model to return key elements with descriptions leveraging Structured Outputs.

## Prerequisites
Make sure you have an API key for the Mistral AI platform. We'll also show you how to load it from environment variables.

In [1]:
# Install the Mistral Python SDK
!pip install mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.4 MB/s eta 0:00:00


## Setup
We'll load the Mistral API key from environment variables and initialize the client. Make sure your API key is saved in your environment variables as `MISTRAL_API_KEY`.


In [2]:
%env MISTRAL_API_KEY=

env: MISTRAL_API_KEY=


In [3]:
import os
from mistralai.client import Mistral

# Load Mistral API key from environment variables
api_key = os.environ["MISTRAL_API_KEY"]

# Model specification
model = "ministral-14b-2512"

# Initialize the Mistral client
client = Mistral(api_key=api_key)


## Data Model
Define a object that will correspond to the expected output of the model. Using structured outputs, we can force the model to answer in a specific format.

In [4]:
from pydantic import BaseModel

class Element(BaseModel):
    name: str
    description: str

class ImageDescription(BaseModel):
    elements: list[Element]
    global_description: str

## Sending Image URL for Description
We'll prompt the model to describe the image by providing an image URL. The response will be returned in a structured JSON format with the key elements described.


In [5]:
# Define the messages for the chat API
messages = [
    {
        "role": "system",
        "content": (
            "You are an image description system. Given an image, describe it by providing a list of elements, their names and descriptions. "
            "And end with a global short description of the full image.\n\n"
            "# Response Format\n"
            "Respond strictly in JSON using the following schema:\n\n"
            "{\n"
            '  "elements": [\n'
            "    {\n"
            '      "name": "<short name of the element>",\n'
            '      "description": "<detailed description of the element>"\n'
            "    }\n"
            "  ],\n"
            '  "global_description": "<short overall description of the image>"\n'
            "}"
        )
    },
    {
        "role": "user",
        "content": "Describe the image"
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": "https://docs.mistral.ai/img/eiffel-tower-paris.jpg"
            }
        ]
    }
]

# Call the Mistral API to complete the chat
chat_response = client.chat.parse(
    model=model,
    messages=messages,
    response_format=ImageDescription
)


## Showing Result
We'll now parse the JSON response from the API. Our SDK parses and provides the object out of the box under`.parsed`, you can access the raw response in `.content`

In [6]:
# Get the object from the response
image_description = chat_response.choices[0].message.parsed

# Print each element and its description
print(image_description.global_description)
for element in image_description.elements:
    print("Element:", element.name)
    print("Description:", element.description, end="\n\n")

# Access the raw content
print(chat_response.choices[0].message.content)

A clear and vivid image of the Eiffel Tower in Paris, showcasing its intricate iron lattice structure against a bright blue sky, with a view of distant buildings and autumn trees at its base.
Element: Eiffel_Tower
Description: An iconic iron lattice tower with intricate wrought-iron framework, featuring four robust support legs that converge at the base and extend upward. The structure includes multiple observation decks at varying heights, connected by an internal staircase and elevator system. The top section houses antennas and other communication equipment.

Element: Base_Arches
Description: The lower portion of the Eiffel Tower, characterized by large, arched supports that provide structural stability. These arches are adorned with decorative elements and form a grand entrance area beneath the tower.

Element: Sky
Description: A clear, deep blue sky with no visible clouds, providing a stark and vibrant backdrop that highlights the tower's intricate ironwork.

Element: Background_B

And done ! You know how to use the Mistral Vision capabilities to describe an image by sending an image URL and receiving a structured JSON response. The descriptions provided by the model offer insights into the key elements of the image.
